In [27]:
%load_ext autoreload
%autoreload 2
import json
from dotenv import load_dotenv
import json
import os

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
load_dotenv()

GIRDER_API_URL = os.getenv("GIRDER_API_URL")
API_KEY = os.getenv("HTMDEC_GIRDER_API_KEY")
ALPSS_FORM_ID = '6931adb5bb4e23b1e86a1ff1'

In [9]:
from girder_client import GirderClient
client = GirderClient(apiUrl=GIRDER_API_URL)
client.authenticate(apiKey=API_KEY)
user = client.get("user/me")
print(f"✅ Authenticated as {user['login']}")

✅ Authenticated as arachid1


In [32]:
igsns = client.get('deposition', parameters={'limit': 10000})
# igsns = igsns[:50]
print("number of igsns: ", len(igsns))

number of igsns:  1988


In [33]:
def analyze_files_for_igsn(client, igsn):
    params = {
        "q": igsn,
        "mode": "igsn",
        "types": '["folder","item"]',
        "limit": 1000,
    }

    resources = client.get("resource/search", parameters=params)

    xrd_count = 0
    ebsd_count = 0

    for resource_type_list in resources.values():
        for resource in resource_type_list:
            name = resource.get("name", "")
            if name.endswith("_xrd.csv"):
                xrd_count += 1
            # elif name.endswith("_ebsd.csv"):
            #     ebsd_count += 1

    return {
        "has_xrd": xrd_count > 0,
        "has_ebsd": ebsd_count > 0,
        "xrd_count": xrd_count,
        "ebsd_count": ebsd_count,
    }

def analyze_laser_shock_for_igsn(client, igsn, form_id):
    results = client.get(
        "entry",
        parameters={
            "formId": form_id,
            "query": igsn,
            "field": "file_name",
            "limit": 100000,
        },
    )

    laser_count = len(results) if results else 0

    return {
        "has_laser_shock": laser_count > 0,
        "laser_shock_count": laser_count,
    }


In [ ]:
from tqdm import tqdm

analysis_results = {}

for dep in tqdm(igsns, desc="Analyzing IGSNs"):
    igsn = dep["igsn"]

    file_info = analyze_files_for_igsn(client, igsn)
    laser_info = analyze_laser_shock_for_igsn(client, igsn, ALPSS_FORM_ID)

    has_any_data = (
        file_info["has_xrd"]
        or file_info["has_ebsd"]
        or laser_info["has_laser_shock"]
    )

    analysis_results[igsn] = {
        "has_any_data": has_any_data,
        **file_info,
        **laser_info,
    }


ModuleNotFoundError: No module named 'tqdm'

In [23]:
summary = {
    "total_igsns": len(analysis_results),
    "igsns_with_any_data": 0,
    "igsns_with_xrd": 0,
    "igsns_with_ebsd": 0,
    "igsns_with_laser_shock": 0,
    "total_xrd_files": 0,
    "total_ebsd_files": 0,
    "total_laser_shock_entries": 0,
}

for data in analysis_results.values():
    if data["has_any_data"]:
        summary["igsns_with_any_data"] += 1
    if data["has_xrd"]:
        summary["igsns_with_xrd"] += 1
    if data["has_ebsd"]:
        summary["igsns_with_ebsd"] += 1
    if data["has_laser_shock"]:
        summary["igsns_with_laser_shock"] += 1

    summary["total_xrd_files"] += data["xrd_count"]
    summary["total_ebsd_files"] += data["ebsd_count"]
    summary["total_laser_shock_entries"] += data["laser_shock_count"]


In [ ]:
print(json.dumps(summary, indent=3))

{
   "total_igsns": 50,
   "igsns_with_any_data": 6,
   "igsns_with_xrd": 6,
   "igsns_with_ebsd": 0,
   "igsns_with_laser_shock": 0,
   "total_xrd_files": 846,
   "total_ebsd_files": 0,
   "total_laser_shock_entries": 0
}


In [ ]:
import pandas as pd
df = (
    pd.DataFrame.from_dict(analysis_results, orient="index")
    .reset_index()
    .rename(columns={"index": "igsn"})
)

df.head()

In [ ]:
df["xrd_count"].value_counts().sort_index()
df["laser_shock_count"].value_counts().sort_index()